In [3]:
# Fast full-dataset model comparison (single cell)
# - Uses full dataset if present (data/raw/creditcard.csv). Falls back to sample.
# - Uses a train/validation/test split (no expensive CV loops).
# - Uses efficient estimators + early stopping and sensible hyperparams.
# - Saves artifacts to repo-root/artifacts/.
import os
import time
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    roc_curve,
    auc,
)
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# optional xgboost import (use if available)
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

import joblib

# ---------- Safety: ensure project-root cwd ----------
cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir("..")
    print("Changed working directory to project root:", Path.cwd())

# ---------- Paths ----------
DATA_RAW = Path("data/raw")
FULL_CSV = DATA_RAW / "creditcard.csv"
SAMPLE_CSV = DATA_RAW / "sample_raw.csv"

ARTIFACTS = Path("artifacts")
MODELS_DIR = ARTIFACTS / "models"
REPORTS_DIR = ARTIFACTS / "reports"
PLOTS_DIR = REPORTS_DIR / "plots"
for p in (ARTIFACTS, MODELS_DIR, REPORTS_DIR, PLOTS_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ---------- Config ----------
RANDOM_STATE = 42
TEST_SIZE = 0.2   # final holdout
VAL_SIZE = 0.25   # fraction of remaining training used as validation (so total val ~= 0.2)
SAMPLE_FALLBACK = True  # if full csv missing, use sample
VERBOSE = True

# ---------- Load dataset ----------
if FULL_CSV.exists():
    df = pd.read_csv(FULL_CSV)
    print("Loaded full dataset:", FULL_CSV, "shape=", df.shape)
elif SAMPLE_CSV.exists() and SAMPLE_FALLBACK:
    df = pd.read_csv(SAMPLE_CSV)
    print("Full dataset not found. Loaded sample:", SAMPLE_CSV, "shape=", df.shape)
else:
    raise FileNotFoundError("No dataset found. Place creditcard.csv in data/raw/ or sample_raw.csv as fallback.")

# Sanity: ensure Class exists and correct dtype
if "Class" not in df.columns:
    raise ValueError("Dataset missing 'Class' column.")
df["Class"] = df["Class"].astype(int)

# ---------- Train / Validation / Test split ----------
# First split off final test set
X_all = df.drop(columns=["Class"])
y_all = df["Class"].astype(int)

# Use stratify to preserve class ratio
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, stratify=y_all, random_state=RANDOM_STATE
)

# Further split trainval into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=VAL_SIZE, stratify=y_trainval, random_state=RANDOM_STATE
)

print("Shapes — train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)
print("Class distribution in train:", np.bincount(y_train) if y_train.nunique()>1 else y_train.value_counts().to_dict())

# ---------- Preprocessing: scale numeric (Time & Amount) + keep rest (features are already PCA V1..V28) ----------
# We'll scale all columns (safe here)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Save scaler for inference
joblib.dump(scaler, MODELS_DIR / "scaler.joblib")

# ---------- Helper: evaluation wrapper ----------
def eval_and_save(name, model, X_val, y_val, X_test, y_test):
    """Compute metrics on val/test, save model and plots, return summary dict."""
    res = {}
    # ensure predict_proba available or approximate
    if hasattr(model, "predict_proba"):
        y_proba_val = model.predict_proba(X_val)[:, 1]
        y_proba_test = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        raw_val = model.decision_function(X_val)
        raw_test = model.decision_function(X_test)
        y_proba_val = (raw_val - raw_val.min()) / (raw_val.max() - raw_val.min() + 1e-9)
        y_proba_test = (raw_test - raw_test.min()) / (raw_test.max() - raw_test.min() + 1e-9)
    else:
        y_proba_val = model.predict(X_val).astype(float)
        y_proba_test = model.predict(X_test).astype(float)

    # Validation metrics at 0.5
    val_pred = (y_proba_val >= 0.5).astype(int)
    test_pred = (y_proba_test >= 0.5).astype(int)

    rpt_val = classification_report(y_val, val_pred, output_dict=True, zero_division=0)
    rpt_test = classification_report(y_test, test_pred, output_dict=True, zero_division=0)
    rocauc_val = roc_auc_score(y_val, y_proba_val) if len(np.unique(y_val))>1 else float("nan")
    rocauc_test = roc_auc_score(y_test, y_proba_test) if len(np.unique(y_test))>1 else float("nan")

    res["name"] = name
    res["val_metrics"] = {
        "precision_1": float(rpt_val.get("1", {}).get("precision", 0.0)),
        "recall_1": float(rpt_val.get("1", {}).get("recall", 0.0)),
        "f1_1": float(rpt_val.get("1", {}).get("f1-score", 0.0)),
        "roc_auc": float(rocauc_val),
    }
    res["test_metrics"] = {
        "precision_1": float(rpt_test.get("1", {}).get("precision", 0.0)),
        "recall_1": float(rpt_test.get("1", {}).get("recall", 0.0)),
        "f1_1": float(rpt_test.get("1", {}).get("f1-score", 0.0)),
        "roc_auc": float(rocauc_test),
    }

    # Save ROC + PR + calibration for test
    # PR
    prec, rec, _ = precision_recall_curve(y_test, y_proba_test)
    ap = average_precision_score(y_test, y_proba_test)
    plt.figure()
    plt.plot(rec, prec, label=f"AP={ap:.4f}")
    plt.title(f"PR Curve - {name}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.legend()
    pr_path = PLOTS_DIR / f"{name}_pr.png"
    plt.savefig(pr_path, bbox_inches="tight", dpi=150); plt.close()

    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba_test)
    rocauc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={rocauc:.4f}")
    plt.plot([0,1],[0,1],"--", color="grey")
    plt.title(f"ROC Curve - {name}")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.legend()
    roc_path = PLOTS_DIR / f"{name}_roc.png"
    plt.savefig(roc_path, bbox_inches="tight", dpi=150); plt.close()

    # Calibration (10 bins)
    try:
        from sklearn.calibration import calibration_curve
        prob_true, prob_pred = calibration_curve(y_test, y_proba_test, n_bins=10, strategy="uniform")
        plt.figure()
        plt.plot(prob_pred, prob_true, "o-", label="Calibration")
        plt.plot([0,1],[0,1],"--", color="grey")
        plt.title(f"Calibration - {name}")
        plt.xlabel("Mean Predicted Prob"); plt.ylabel("Fraction Positive")
        plt.legend()
        cal_path = PLOTS_DIR / f"{name}_cal.png"
        plt.savefig(cal_path, bbox_inches="tight", dpi=150); plt.close()
    except Exception:
        cal_path = None

    # Save model artifact
    model_path = MODELS_DIR / f"{name}.joblib"
    try:
        joblib.dump(model, model_path)
    except Exception as e:
        print("Warning: failed saving model:", e)

    res["artifacts"] = {"model_path": str(model_path), "pr_path": str(pr_path), "roc_path": str(roc_path), "cal_path": str(cal_path) if cal_path else None}
    return res

# ---------- Model training with fast configs ----------
start_total = time.time()
results = []

# 1) Logistic Regression (fast baseline) with class_weight
print("Training: LogisticRegression (fast)")
t0 = time.time()
lr = LogisticRegression(max_iter=1000, solver="saga", class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)
lr.fit(X_train_scaled, y_train)
results.append(eval_and_save("logreg_balanced", lr, X_val_scaled, y_val, X_test_scaled, y_test))
print("Done LR in", time.time()-t0, "s")

# 2) HistGradientBoosting (fast, GPU-like algorithm in sklearn; very efficient)
print("Training: HistGradientBoostingClassifier")
t0 = time.time()
hgb = HistGradientBoostingClassifier(max_iter=300, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=RANDOM_STATE)
hgb.fit(X_train_scaled, y_train)
results.append(eval_and_save("hgb", hgb, X_val_scaled, y_val, X_test_scaled, y_test))
print("Done HGB in", time.time()-t0, "s")

# Robust XGBoost training block (handles different xgboost versions)
import inspect
t0 = time.time()
pos = int((y_train == 1).sum())
neg = int((y_train == 0).sum())
scale_pos_weight = max(1, int(neg / max(1, pos)))

# choose initial n_estimators smaller when fallback
n_estimators_full = 500
n_estimators_fallback = 200

xgb = XGBClassifier(
    n_estimators=n_estimators_full,
    max_depth=6,
    learning_rate=0.05,
    tree_method="hist",  # fast on tabular data
    use_label_encoder=False,
    eval_metric="auc",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
)

# Try to detect if fit supports early_stopping_rounds
supports_es = False
try:
    sig = inspect.signature(xgb.fit)
    if "early_stopping_rounds" in sig.parameters:
        supports_es = True
except Exception:
    # fallback: assume not supported
    supports_es = False

if supports_es:
    print("XGBoost fit supports early_stopping_rounds -> using early stopping.")
    try:
        xgb.fit(X_train_scaled, y_train, eval_set=[(X_val_scaled, y_val)], early_stopping_rounds=25, verbose=False)
    except TypeError as e:
        # defensive: some builds raise TypeError despite signature; fallback below
        print("Warning: unexpected TypeError when using early_stopping_rounds:", e)
        supports_es = False

if not supports_es:
    print("XGBoost fit does NOT support early_stopping_rounds in this environment.")
    print(f"Falling back to fit without early stopping (n_estimators={n_estimators_fallback}) for speed.")
    xgb.set_params(n_estimators=n_estimators_fallback)
    xgb.fit(X_train_scaled, y_train)

results.append(eval_and_save("xgb_hist", xgb, X_val_scaled, y_val, X_test_scaled, y_test))
print("Done XGB in", time.time() - t0, "s")

# 4) Random Forest with restrained complexity
print("Training: RandomForest (restricted depth for speed)")
t0 = time.time()
rf = RandomForestClassifier(n_estimators=200, max_depth=14, min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
results.append(eval_and_save("rf_restricted", rf, X_val_scaled, y_val, X_test_scaled, y_test))
print("Done RF in", time.time()-t0, "s")

# ---------- Save summary results ----------
summary_df = pd.DataFrame([{"model": r["name"], **{f"test_{k}": v for k,v in r["test_metrics"].items()}} for r in results])
summary_csv = REPORTS_DIR / "model_comparison_summary.csv"
summary_df.to_csv(summary_csv, index=False)
# save full JSON details
with open(REPORTS_DIR / "model_comparison_details.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved comparison summary to:", summary_csv)
print(summary_df)

print("Total script time: {:.1f}s".format(time.time() - start_total))

Loaded full dataset: data/raw/creditcard.csv shape= (284807, 31)
Shapes — train: (170883, 30) val: (56962, 30) test: (56962, 30)
Class distribution in train: [170588    295]
Training: LogisticRegression (fast)


/Users/apple/TechStack/Projects/Creditcard_Fraud_Detection/venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Done LR in 43.14089298248291 s
Training: HistGradientBoostingClassifier
Done HGB in 1.0277938842773438 s
XGBoost fit does NOT support early_stopping_rounds in this environment.
Falling back to fit without early stopping (n_estimators=200) for speed.


/Users/apple/TechStack/Projects/Creditcard_Fraud_Detection/venv/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [02:25:13] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Done XGB in 1.4694812297821045 s
Training: RandomForest (restricted depth for speed)
Done RF in 18.38158416748047 s
Saved comparison summary to: artifacts/reports/model_comparison_summary.csv
             model  test_precision_1  test_recall_1  test_f1_1  test_roc_auc
0  logreg_balanced          0.063617       0.908163   0.118904      0.971576
1              hgb          0.510638       0.244898   0.331034      0.312214
2         xgb_hist          0.855670       0.846939   0.851282      0.971231
3    rf_restricted          0.873563       0.775510   0.821622      0.963629
Total script time: 64.0s
